In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

# Example input
input_text = "The capital of France is"
inputs = tokenizer(input_text, return_tensors="pt")

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits  # Shape: (batch_size, seq_len, vocab_size)

# Get token distribution (softmax over the last token)
last_token_logits = logits[0, -1]  # Shape: (vocab_size,)
probs = torch.softmax(last_token_logits, dim=-1)

# Top 5 token predictions
top_probs, top_indices = probs.topk(5)
top_tokens = tokenizer.convert_ids_to_tokens(top_indices)
print(list(zip(top_tokens, top_probs.tolist())))


/Users/ian/Programming/general_knowledge_distillation/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[('▁Paris', 0.5089597702026367), ('▁located', 0.16609974205493927), (':', 0.02175397053360939), ('▁the', 0.02156653255224228), ('?', 0.020989036187529564)]


# General Knowledge Distillation

Create dataset with TextDataset class -> GKDTrainer.
All functions will be stored in a GKD file which we will import.

Text Dataset class

In [ ]:
from torch.utils.data import DataLoader, Dataset

# class TextDataset(Dataset):
#     def __init__(self, dataframe, tokenizer, max_length):
#         self.dataframe = dataframe
#         self.tokenizer = tokenizer
#         self.max_length = max_length

#     def __len__(self):
#         return len(self.dataframe)
    
#     def __getitem__(self, index):
#         row = self.dataframe.iloc[index]
#         input_tokens = self.tokenizer( 
#             row['prompt'], 
#             truncation=True, 
#             padding='max_length', 
#             max_length=self.max_length, 
#             return_tensors='pt' #  return the output in the form of PyTorch tensors
#             )
#         return {
#             'input_ids': input_tokens['input_ids'].squeeze(0),
#             'attention_mask': input_tokens['attention_mask'].squeeze(0),
#             'labels': torch.tensor(row['relevant'], dtype=torch.float), # for BCEWithLogitsLoss use float
        # }

# need to customize further probably
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Example: texts[idx] is "Prompt: ... Completion: ..."
        full_text = self.texts[idx]
        tokens = self.tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        input_ids = tokens['input_ids'].squeeze(0)
        attention_mask = tokens['attention_mask'].squeeze(0)

        # We can handle masking with prompt length or other methods like a distinct seperator
        prompt_length = self.get_prompt_length(full_text)

        labels = input_ids.clone()
        labels[:prompt_length] = -100  # mask prompt tokens in labels

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }
    
dataset = TextDataset(["text1", "text2", "text3"], tokenizer)

In [ ]:
import torch
import bitsandbytes as bnb
import numpy as np
from torch.utils.data import DataLoader, RandomSampler
import torch.nn.functional as F

class GKDTrainer():
    def __init__(
            self, model, teacher_model, optimizer, scheduler, train_dataset, steps, batch_size, temperature=1,
            eval_dataset=None, steps_between_val=None, shuffle=True, student_data_fraction=0.5, sampling_method='sample', divergence='KL', 
            learning_rate=1e-4, device='cuda'
    ):
        self.model = model
        self.teacher_model = teacher_model 
        self.train_dataset = train_dataset
        self.temperature = temperature # temperature of the teacher's distribution
        self.eval_dataset = eval_dataset
        self.steps_between_val = (steps_between_val if steps_between_val is not None else steps) # if steps_between_val is not set, then we initialize self.steps_between_val to steps
        self.student_data_fraction = student_data_fraction # probability that we perform on policy KD each step
        self.sampling_method = sampling_method # the method with which we sample tokens from the student's output distribution when doing on policy KD
        self.divergence = divergence # measure of the difference between the teacher and student output distribution i.e. the loss
        self.learning_rate=learning_rate
        self.optimizer = optimizer # probably can use one of the paged optimizers from https://huggingface.co/docs/bitsandbytes/main/en/reference/optim/adamw
        self.scheduler = scheduler
        self.steps = steps # number of gradient updates
        self.device = device # ensure device is set to 'mps' if running on mac

        sampler = RandomSampler(self.train_dataset, replacement=True, num_samples=(self.steps * batch_size)) # (self.steps * batch_size) because for each batch we do one weight update and hence one step 
        self.train_dataloader = DataLoader(self.train_dataset, batch_size=batch_size, sampler=sampler)

        # Add validation dataloader here

        if self.divergence == 'KL' or self.divergence == 'reverse KL':
            self.loss_fn = torch.nn.KLDivLoss(
                reduction='batchmean',
                log_target=True # whether the target values are in the log space
            )

        elif self.divergence == 'JSD':
            # implement later
            pass

        else:
            raise ValueError(
            f"{self.divergence} is not an available divergence. Choose a divergence from this list: ['KL', 'reverse KL', 'JSD']"
        )

        # prepare for training
        self.model.to(self.device)
        self.teacher_model.to(self.device)
        self.model.train()
        self.teacher_model.eval()

    def train(self):

        for i, batch in enumerate(self.train_dataloader):
            p = np.random.uniform(0, 1)

            # move data to the gpu
            inputs = {
                'input_ids': batch['input_ids'].to(self.device),
                'attention_mask': batch['attention_mask'].to(self.device),
            }

            labels = batch['labels'].to(self.device)

            if p <= self.student_data_fraction: # on policy KD 
                predicted_logits = self.model(**inputs).logits
                
                # sample from the student's output distribution with sampling_method
                if self.sampling_method == 'sample': # normal sampling
                    probs = torch.softmax(predicted_logits, dim=-1)
                    predicted_token_ids = torch.multinomial(probs.view(-1, probs.size(-1)), 1)
                    predicted_token_ids = predicted_token_ids.view(predicted_logits.shape[:-1]) 
                    pass
            
                elif self.sampling_method == 'greedy': # simply taking the token with the largest logit
                    predicted_token_ids = torch.argmax(predicted_logits, dim=-1)  # shape: [batch_size, seq_len]

                elif self.sampling_method == 'top-k':
                    # implement later
                    pass 

                elif self.sampling_method == 'top-p':
                    # implement later
                    pass 


                with torch.no_grad():
                    target_logits = self.teacher_model(input_ids=predicted_token_ids).logits / self.temperature

            else: # supervised KD 

                # no need for gradients for the teacher 
                with torch.no_grad(): 
                    target_logits = self.teacher_model(**inputs).logits / self.temperature
                # predicted_logits = self.model(**inputs).logits / self.temperature
                predicted_logits = self.model(**inputs).logits

            # obtain log probabilites
            predicted_log_probs = F.log_softmax(predicted_logits, dim=-1)
            target_log_probs = F.log_softmax(target_logits, dim=-1)

            # flatten batch and sequence dims for masking
            batch_size, seq_len, vocab_size = predicted_logits.shape
            predicted_log_probs = predicted_log_probs.view(-1, vocab_size) 
            target_log_probs = target_log_probs.view(-1, vocab_size)         

            labels_flat = labels.view(-1)
            mask = labels_flat != -100
                                    
            # mask out padded tokens
            predicted_log_probs_masked = predicted_log_probs[mask]
            target_log_probs_masked = target_log_probs[mask]

            # calculate loss
            if self.divergence == 'KL':
                loss = self.loss_fn(predicted_log_probs_masked, target_log_probs_masked)
            elif self.divergence == 'reverse KL':
                loss = self.loss_fn(target_log_probs_masked, predicted_log_probs_masked)
            else:
                pass

            # backpropagate
            self.optimizer.zero_grad() 
            loss.backward()
            self.optimizer.step()

            # validation
            if (((i + 1) % self.steps_between_val) == 0) and self.eval_dataset is not None:
                with torch.no_grad():
                    # validation code here
                    pass

            # scheduler
            # scheduler step here
